In [1]:
import numpy as np
import pandas as pd
from modules.sumo import check_feasibility,update_green_times,SumoSimulation,update_green_time_dict,connexions_routes_correspondance_matrix,ConnexionsGreenTime
import sumolib
import xml.etree.ElementTree as ET
import torch
import sympy as sp
from torch.quasirandom import SobolEngine
import argparse
from modules.sumo import create_gt_dataset
from modules.sumo import TLIncomingCorridors
from modules.sumo import CompleteCorridorsConnexions
from itertools import groupby


In [23]:
net_xml="./ingolstadt7/ingolstadt7.net.xml"
routes_xml="./ingolstadt7/ingolstadt7_routes_with_edges.rou.xml"
corridors_generator=TLIncomingCorridors(net_xml,routes_xml)
corridors_df=corridors_generator.get_corridors_df()
complete_corridors_connexions_generator=CompleteCorridorsConnexions(routes_xml,net_xml,corridors_df)

complete_corridors_connexions_df=complete_corridors_connexions_generator.create_connexion_corridors_df()

In [25]:
complete_corridors_connexions_df.columns

Index(['connexion_id', 'tl', 'route', 'route_last_edge', 'route_first_edge',
       'route_length_ff_tt', 'route_sum_length', 'route_sum_ff_tt',
       'route_ff_speed', 'origin', 'route_n_edges', 'route_str',
       'route_str_str', 'route_demand', 'tl_demand', 'tl_n_routes',
       'tl_n_edges', 'add_info', 'n_previous_tl', 'n_previous_routes',
       'n_previous_edges', 'previous_demand', 'next_info', 'direct_next_edges',
       'direct_next_tls', 'from_info', 'direct_from_edges',
       'direct_previous_tls', 'direct_next_demand', 'direct_previous_demand',
       'direct_from_n_junction_inc_lanes', 'route_id', 'to_edge', 'link_index',
       'green_phases', 'connexion_demand', 'route_connexion_demand'],
      dtype='str')

In [10]:
tree = ET.parse("ingolstadt7/gt_data/gt_0_InTAS7DIM.vehroute.xml")
root = tree.getroot()

data = []

for vehicle in root.findall("vehicle"):
    vehicle_id = vehicle.get("id")
    depart_tim=vehicle.get("depart")

    for route in vehicle.findall("route"):
        data.append({
            "vehicle_id": vehicle_id,
            "depart":vehicle.get("depart"),
            "route": route.get("edges").split(" "),
            "exit_times": [float(x) for x in route.get("exitTimes").split(" ")]
        })

df = pd.DataFrame(data)

df["route_str"]=df["route"].astype(str)
df["route_str_str"]=df["route"].str.join(" ")
df=df[~df["exit_times"].apply(lambda x : -1 in x)]
display(df)

,vehicle_id,depart,route,exit_times,route_str,route_str_str
0,carIn89578:1,57614.00,"[124812856#0, 124812856#1, 201956810]","[57620.0, 57621.0, 57629.0]","['124812856#0', '124812856#1', '201956810']",124812856#0 124812856#1 201956810
1,carIn107084:1,57601.00,"[124812856#0, 124812856#1, 201956821#0, 201956...","[57607.0, 57608.0, 57614.0, 57616.0, 57629.0, ...","['124812856#0', '124812856#1', '201956821#0', ...",124812856#0 124812856#1 201956821#0 201956821#...
2,carIn105842:1,57601.00,"[653473569#5, 164051413, 124812857#0, 20195681...","[57610.0, 57612.0, 57624.0, 57647.0]","['653473569#5', '164051413', '124812857#0', '2...",653473569#5 164051413 124812857#0 201956811#0
3,randUni10474:1,57605.00,"[124812856#0, 124812856#1, 201956821#0, 201956...","[57611.0, 57612.0, 57619.0, 57622.0, 57638.0, ...","['124812856#0', '124812856#1', '201956821#0', ...",124812856#0 124812856#1 201956821#0 201956821#...
4,carIn106322:1,57614.00,"[653473569#5, 164051413, 124812857#0, 20195681...","[57623.0, 57625.0, 57639.0, 57649.0]","['653473569#5', '164051413', '124812857#0', '2...",653473569#5 164051413 124812857#0 201956811#0
...,...,...,...,...,...,...
2904,h6750c1:1,61139.00,"[315358253#1, 315358253#2, 402600768#0, 402600...","[61147.0, 61163.0, 61168.0, 61173.0, 61176.0, ...","['315358253#1', '315358253#2', '402600768#0', ...",315358253#1 315358253#2 402600768#0 402600768#...
2905,h6909c1:1,61142.00,"[315358253#1, 315358253#2, 402600768#0, 402600...","[61150.0, 61166.0, 61170.0, 61175.0, 61178.0, ...","['315358253#1', '315358253#2', '402600768#0', ...",315358253#1 315358253#2 402600768#0 402600768#...
2906,randUni29386:1,61179.00,"[-173169611#0, 201956820]","[61188.0, 61194.0]","['-173169611#0', '201956820']",-173169611#0 201956820
2907,h7056c1:1,61145.00,"[315358253#1, 315358253#2, 402600768#0, 402600...","[61154.0, 61168.0, 61172.0, 61177.0, 61179.0, ...","['315358253#1', '315358253#2', '402600768#0', ...",315358253#1 315358253#2 402600768#0 402600768#...


In [11]:
def average_route_traveltime(row,vehroute_df):
    route=row["route"]+[row["to_edge"]]
    route_str_str=" ".join(route)

    vehroute_df_route=vehroute_df[vehroute_df["route_str_str"].str.contains(route_str_str,regex=False)]


    if vehroute_df_route.empty:
        return np.nan

    def extract_times_index(vehroute_df_route_row,sub_route):
        full_route=vehroute_df_route_row["route"]
        start_indices = [i for i in range(len(full_route) - len(sub_route) + 1) if full_route[i:i+len(sub_route)] == sub_route]

        
        end_idices = [start_indx + len(sub_route) - 1 for start_indx in start_indices]

        return (start_indices,end_idices)

    vehroute_df_route["indices"]=vehroute_df_route.apply(extract_times_index,args=(route,),axis=1)

    vehroute_df_route["start_indices"] = vehroute_df_route["indices"].str[0]
    vehroute_df_route["end_indices"] = vehroute_df_route["indices"].str[1]

    vehroute_df_route["route_depart_times"]=vehroute_df_route.apply(
        lambda x : 
        [float(x["exit_times"][start_idx-1]) if start_idx>0 else float(x["depart"]) for start_idx in x["start_indices"]],
        axis=1)

    vehroute_df_route["route_exit_times"]=vehroute_df_route.apply(
        lambda x : [float(x["exit_times"][end_indx-1]) for end_indx in x["end_indices"]],
        axis=1)

    vehroute_df_route["route_traveltime"]=vehroute_df_route.apply(
        lambda x : [exit_time-depart_time for exit_time,depart_time in zip(x["route_exit_times"],x["route_depart_times"])],
        axis=1
    )


    traveltime_values=[tt for tt_list in vehroute_df_route["route_traveltime"].values for tt in tt_list]
    average_route_traveltime=np.mean(traveltime_values)


    if len(vehroute_df_route) == 0 or np.isnan(average_route_traveltime):
        print(f"connexion id : {row['connexion_id']}")
        print("Nombre de lignes :", len(vehroute_df_route))
        print("Average traveltime :",average_route_traveltime)
        print("-------------------------------------------------------------------")

    return average_route_traveltime
    

In [12]:
complete_corridors_connexions_df["route_average_traveltime"]=complete_corridors_connexions_df.apply(average_route_traveltime,args=(df,),axis=1)

In [13]:
print(len(complete_corridors_connexions_df[complete_corridors_connexions_df["route_average_traveltime"].isna()]))

8


In [29]:
print(len(complete_corridors_connexions_df["connexion_id"]))
print(len(complete_corridors_connexions_df["connexion_id"].unique()))
print(len(complete_corridors_connexions_df))

91
91
91


In [ ]:
!python gt_dataset_generation_script.py

In [14]:
import pandas as pd
df_gt=pd.read_csv("./gt_data/df_gt_InTAS7DIM.csv")

In [18]:
df_gt

,Unnamed: 0,connexion_id,route_connexion_average_traveltime,tl,route,route_str,route_str_str,route_last_edge,route_first_edge,route_demand,route_id,to_edge,link_index,connexion_demand,green_phases,route_connexion_demand
0,0,0,19.469767,cluster_1757124350_1757124352,['-173169611#0'],['-173169611#0'],-173169611#0,-173169611#0,-173169611#0,78,0,201956820,3,44,['rrrGGGrr'],78
1,1,1,27.405882,cluster_1757124350_1757124352,['-173169611#0'],['-173169611#0'],-173169611#0,-173169611#0,-173169611#0,78,0,201956821#0,4,34,['rrrGGGrr'],78
2,2,2,22.610765,cluster_1757124350_1757124352,"['124812856#0', '124812856#1']","['124812856#0', '124812856#1']",124812856#0 124812856#1,124812856#1,124812856#0,656,1,201956821#0,0,527,"['GGgrrGGG', 'GGGrrrrr']",1185
3,3,3,22.610765,cluster_1757124350_1757124352,"['124812856#0', '124812856#1']","['124812856#0', '124812856#1']",124812856#0 124812856#1,124812856#1,124812856#0,656,1,201956821#0,1,527,"['GGgrrGGG', 'GGGrrrrr']",1185
4,4,4,30.509311,cluster_1757124350_1757124352,"['124812856#0', '124812856#1']","['124812856#0', '124812856#1']",124812856#0 124812856#1,124812856#1,124812856#0,656,1,201956810,2,131,"['GGgrrGGG', 'yygrryyy', 'GGGrrrrr']",1185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,76,85,137.340713,gneJ207,['201963537#1'],['201963537#1'],201963537#1,201963537#1,201963537#1,797,1,-164051413,2,404,"['GGgGrGGG', 'yygyryyy', 'GGGrrrrr']",1188
77,77,87,54.873553,gneJ207,"['25145012#7', '104010354']","['25145012#7', '104010354']",25145012#7 104010354,104010354,25145012#7,110,2,124812857#0,6,420,['GGgGrGGG'],887
78,78,88,54.873553,gneJ207,"['25145012#7', '104010354']","['25145012#7', '104010354']",25145012#7 104010354,104010354,25145012#7,110,2,124812857#0,7,420,['GGgGrGGG'],887
79,79,89,12.486526,gneJ207,"['653473569#5', '164051413']","['653473569#5', '164051413']",653473569#5 164051413,164051413,653473569#5,394,3,124812857#0,3,304,"['GGgGrGGG', 'rrrGGGrr']",394


In [5]:
df_all=None
for rep in range(10):
    df=pd.read_csv(f"./gt_data/gt_{rep}_InTAS7DIM.travel_times.csv")
    if rep==0:
        df_all=df
    else:
        df_all=pd.concat([df_all,df],axis=0)

In [18]:
df_gt["route_connexion_average_traveltime"].isna().any()

np.False_

In [3]:
X_DIM=7

def get_tls():
    green_phase_times_dict_tmp = update_green_time_dict(33*np.ones(X_DIM))
    tl_ids_set = []
    for elem in green_phase_times_dict_tmp.keys():
        if elem[0] not in tl_ids_set:
            tl_ids_set.append(elem[0])
    return tl_ids_set   

TL_IDS_SET = get_tls()


net_xml="./ingolstadt7/ingolstadt7.net.xml"
route_xml="./ingolstadt7/ingolstadt7_routes_with_edges.rou.xml"

corridors_generator=TLIncomingCorridors(net_xml,route_xml)
corridors_df=corridors_generator.get_corridors_df()
g_dict_to_vary={
        "gneJ260":"GGGGGgrrr",
        "gneJ210": "GGggrrrrrrGGGG",
        "32564122": "GGGGGgrrr",
        "gneJ207":"GGgGrGGG",
        "gneJ143": "rrrGGGGgGGGg",
        "cluster_1757124350_1757124352":"GGgrrGGG",
        "cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190":"GGGGGGrrrrrr"
    }

g_dict_to_change={
        "gneJ260":"GrrrrrGGG",
        "gneJ210": "rrrrGGGGGGGGrr",
        "32564122": "GrrrrrGGG",
        "gneJ207":"rrrGGGrr",
        "gneJ143": "GGGGrrrrrrrr",
        "cluster_1757124350_1757124352":"rrrGGGrr",
        "cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190":"rrrrrrrrGGGG"
    }

g_dict_const={
    "gneJ260":["yyyyygrrr","rrrGGGrrr","rrryyyrrr","yrrrrryyy"],
    "gneJ210": ["yyggrrrrrryyyy","GGGGrrrrrrrrrr","yyyyrrrrrrrrrr","rrrryyyyyyyyrr"],
    "32564122": ["yyyyyyrrr","yrrrrryyy"],
    "gneJ207":["yygyryyy","GGGrrrrr","yyyrrrrr","rrryyyrr"],
    "gneJ143": ["rrryyyygyyyg","rrrrrrrGrrrG","rrrrrrryrrry","yyyyrrrrrrrr"],
    "cluster_1757124350_1757124352":["yygrryyy","GGGrrrrr","yyyrrrrr","rrryyyrr"],
    "cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190":["rrrrrrrrGGyy","rrrrrrGGGGrr","rrrrGGGGGGrr","rrrrGGyyyyrr","yyyyyyrrrrrr"]
}

dtype=torch.double

complete_corridors_connexions_generator=CompleteCorridorsConnexions(route_xml,net_xml,corridors_df)

complete_corridors_connexions_df=complete_corridors_connexions_generator.create_connexion_corridors_df()

connexions_routes_correspondance_tensor,connexions_routes_correspondance_df=connexions_routes_correspondance_matrix(complete_corridors_connexions_df)

connexions_green_time_creator=ConnexionsGreenTime(
    tl_ids=TL_IDS_SET,
    complete_corridors_connexions_df=complete_corridors_connexions_df,
    net_xml=net_xml,
    g_dict_to_change=g_dict_to_change,
    g_dict_to_vary=g_dict_to_vary,
    g_dict_const=g_dict_const,
    dtype=dtype
)